In [2]:
import polars as pl
from pathlib import Path

trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
gdrive_path = trec_root / "data/gdrive/data"
official_path = trec_root / "data/official"
output_path = trec_root / "results/rerank"


run_path = f"{gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run"
rerank_part_path = f"{dataset_root}/reranked/v2.1/bge-m3-knn-k15/ppr"

rerankdf = pl.read_parquet(f"{rerank_part_path}/*.parquet")
rerankdf.collect_schema()
rerankdf.head()

qid,Q0,docid,rank,score,run_name
i64,str,i64,i64,f64,str
2001,"""Q0""",58865,1,0.030652,"""gemini-25_alias"""
2001,"""Q0""",73331355,4,0.017534,"""gemini-25_alias"""
2001,"""Q0""",4907816,2,0.01724,"""gemini-25_alias"""
2001,"""Q0""",43936047,6,0.015191,"""gemini-25_alias"""
2001,"""Q0""",64323316,7,0.014857,"""gemini-25_alias"""


In [6]:
# write out the reranked result to a trec styled file
reranked_path = f"{output_path}/gemini-2.5-flash-dev3-ppr-knn-k15-v2.1/results.txt"
Path(reranked_path).parent.mkdir(parents=True, exist_ok=True)
# space as separator, no index, no header
rerankdf.write_csv(reranked_path, include_header=False, separator=" ")
! head {reranked_path}

2001 Q0 58865 1 0.030652279821040757 gemini-25_alias
2001 Q0 73331355 4 0.017534025716264303 gemini-25_alias
2001 Q0 4907816 2 0.01723999634359985 gemini-25_alias
2001 Q0 43936047 6 0.015191143542767552 gemini-25_alias
2001 Q0 64323316 7 0.014856797888691556 gemini-25_alias
2001 Q0 7561924 10 0.014531870187688142 gemini-25_alias
2001 Q0 53500318 11 0.01451251748723914 gemini-25_alias
2001 Q0 49363330 3 0.014148206164534459 gemini-25_alias
2001 Q0 37845530 5 0.014073913940429394 gemini-25_alias
2001 Q0 1650010 8 0.01371196671925096 gemini-25_alias


In [7]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    -c \
    {official_path}/dev3-2025-qrel.txt \
    {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run

recip_rank            	all	0.3001
recall_1000           	all	0.4403
ndcg_cut_10           	all	0.3267
ndcg_cut_1000         	all	0.3327


In [8]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    -c \
    {official_path}/dev3-2025-qrel.txt \
    {reranked_path} 

recip_rank            	all	0.1173
recall_1000           	all	0.4403
ndcg_cut_10           	all	0.1551
ndcg_cut_1000         	all	0.1867
